# Module 31 — Debate, and Swarm-style handoff

**THE ONE IDEA:** two more coordination shapes, at opposite ends of the cost curve.

**Debate** — proposer vs critic, with a judge. Highest accuracy on contestable questions,
**2-5x the cost**, and it needs a judge that is actually better than the debaters.

**Handoff** — the Swarm primitive. A handoff is just **a tool call that returns a
different agent**. Cheapest possible multi-agent: no supervisor, no extra reasoning turn,
the conversation simply continues with someone else.

Closes Block J. Coordination patterns, cheapest first: **pipeline → handoff → supervisor
→ debate.**


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import json
from _providers import get_client
from _tools import openai_schemas, run_tool

client, MODEL, _ = get_client("openai")
def ask(prompt, system=None, max_tok=320):
    msgs = ([{"role": "system", "content": system}] if system else [])
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(model=MODEL, max_tokens=max_tok, messages=msgs)
    return r.choices[0].message.content.strip()

# Contestable on purpose. A lookup question would make debate pointless.
QUESTION = ("A self-employed applicant with 2 years of accounts wants 92% LTV on a "
            "flat above a commercial unit. Should this be approved?")
POLICY = "\n".join(run_tool("search_policy", {"query": k})
                   for k in ["ltv", "income", "deposit"])

## Debate — proposer, critic, judge

In [ ]:
prop = ask(f"POLICY:\n{POLICY}\n\n{QUESTION}\nArgue FOR approval in 3 sentences.",
           system="You are the applicant's broker. Make the strongest honest case.")
crit = ask(f"POLICY:\n{POLICY}\n\n{QUESTION}\nPROPOSAL: {prop}\n"
           "Attack it in 3 sentences. Cite the specific policy conflicts.",
           system="You are a credit-risk reviewer. Find what the broker glossed over.")
judge = ask(f"POLICY:\n{POLICY}\n\nQUESTION: {QUESTION}\n\nFOR: {prop}\n\nAGAINST: {crit}\n"
            "Rule: APPROVE, DECLINE or REFER, and one sentence of reasoning.",
            system="You are the underwriting committee. Decide.")

for tag, txt in [("PROPOSER", prop), ("CRITIC", crit), ("JUDGE", judge)]:
    print(f"\n[{tag}] {txt[:230]}")

## Single-agent control

The same question, one call. Compare what the debate actually surfaced.

In [ ]:
solo = ask(f"POLICY:\n{POLICY}\n\n{QUESTION}\nAPPROVE, DECLINE or REFER, one sentence why.")
print("SOLO  :", solo[:220])
print("\nDEBATE:", judge[:220])
print("\n3-5 LLM calls vs 1. Ask whether the extra ones changed the DECISION or")
print("just the prose around it — on many questions it is the latter.")

## Handoff — a tool call that returns an agent

In [ ]:
AGENTS = {
 "triage":   dict(sys="Route the customer. Use a transfer tool. Do not answer yourself.",
                  tools=["search_policy"]),
 "policy":   dict(sys="You answer bank-policy questions using search_policy.",
                  tools=["search_policy"]),
 "payments": dict(sys="You handle payment arithmetic using calculate.",
                  tools=["calculate"]),
}
HANDOFFS = {"transfer_to_policy": "policy", "transfer_to_payments": "payments"}

def handoff_schemas():
    return [{"type": "function", "function": {"name": n,
             "description": f"Transfer this conversation to the {t} specialist.",
             "parameters": {"type": "object", "properties": {}, "required": []}}}
            for n, t in HANDOFFS.items()]

def run_swarm(user_msg, start="triage", max_hops=4):
    agent, msgs, path = start, [{"role": "user", "content": user_msg}], [start]
    for _ in range(max_hops):
        cfg = AGENTS[agent]
        r = client.chat.completions.create(model=MODEL, max_tokens=400,
                messages=[{"role": "system", "content": cfg["sys"]}] + msgs,
                tools=openai_schemas(cfg["tools"]) + (handoff_schemas() if agent == "triage" else []))
        m = r.choices[0].message
        if r.choices[0].finish_reason != "tool_calls":
            return m.content, path
        msgs.append(m)
        for tc in m.tool_calls:
            if tc.function.name in HANDOFFS:
                agent = HANDOFFS[tc.function.name]; path.append(agent)
                msgs.append({"role": "tool", "tool_call_id": tc.id,
                             "content": f"Transferred to {agent}."})
            else:
                msgs.append({"role": "tool", "tool_call_id": tc.id,
                             "content": run_tool(tc.function.name,
                                                 json.loads(tc.function.arguments))})
    return None, path

ans, path = run_swarm("What is the maximum LTV for a first-time buyer?")
print("path  :", " -> ".join(path))
print("answer:", str(ans)[:170])

print("""
LESSON - four coordination shapes, cheapest first. Pick by what you need, not by
what sounds advanced:

  PIPELINE     (29)  fixed A->B->C. Deterministic, cheapest, errors compound.
  HANDOFF      (31)  a tool call returns a different agent. The conversation
                     CONTINUES - no supervisor, no extra reasoning turn, no
                     re-loaded context. Ideal for support-style routing.
  SUPERVISOR   (30)  a routing LLM call per cycle. Dynamic, auditable, 5-10x.
  DEBATE       (31)  proposer + critic + judge. Best on CONTESTABLE questions,
                     2-5x, and only as good as the judge.

Two things to say out loud about debate: it needs a judge at least as strong as
the debaters, or you have paid 3x to have the wrong answer confidently picked;
and on a LOOKUP question it adds nothing, because there is nothing to contest.

The handoff is the underrated one. It is barely more code than a single agent,
and it solves the real problem multi-agent exists for - tool-surface bloat -
without paying for a supervisor.""")

---

**Next:** Block K — `../K_mcp/32_mcp_server_capabilities.ipynb`